# Medical Named Entity Recognition (NER) Lab

In this lab, we will train a Named Entity Recognition model to identify medical entities (diseases and chemicals) using the BC5CDR dataset and PubMedBERT model.

## 1. Setup and Imports

In [1]:
# Install required packages
!pip install transformers datasets seqeval scikit-learn accelerate -q

# 设置 HuggingFace 国内镜像站
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print("已设置 HuggingFace 镜像: https://hf-mirror.com")

已设置 HuggingFace 镜像: https://hf-mirror.com


In [2]:
import os
# 确保 HuggingFace 镜像设置（需要在 import transformers 之前）
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertForTokenClassification, Trainer, TrainingArguments
from transformers import DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from collections import defaultdict
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. BIO Encoding

BIO (Beginning, Inside, Outside) encoding is used for NER tasks:
- **B-{entity}**: Beginning of an entity
- **I-{entity}**: Inside (continuation) of an entity  
- **O**: Outside (not an entity)

For our medical NER task, we have:
- B-Disease, I-Disease (disease entities)
- B-Chemical, I-Chemical (chemical entities)
- O (not an entity)

In [3]:
# Define label mapping
LABEL_LIST = ['O', 'B-Disease', 'I-Disease', 'B-Chemical', 'I-Chemical']
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

print("Label mapping:")
for label, idx in LABEL2ID.items():
    print(f"  {label}: {idx}")

Label mapping:
  O: 0
  B-Disease: 1
  I-Disease: 2
  B-Chemical: 3
  I-Chemical: 4


## 3. Parse BC5CDR Dataset

The BC5CDR dataset format:
```
6794356|t|Tricuspid valve regurgitation and lithium carbonate toxicity...
6794356|a|A newborn with massive tricuspid regurgitation...
6794356	0	29	Tricuspid valve regurgitation	Disease	D014262
```

- First line: `id|t|title`
- Second line: `id|a|abstract`
- Following lines: `id\tstart\tend\ttext\tentity_type\tontology_id`

In [4]:
def parse_bc5cdr_file(filepath):
    """
    Parse BC5CDR format file and return list of documents with entities.
    
    Returns:
        List of dicts: [{'id': str, 'text': str, 'entities': [(start, end, type), ...]}]
    """
    documents = {}
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
                
            if '|t|' in line:
                # Title line
                parts = line.split('|t|')
                doc_id = parts[0]
                title = parts[1] if len(parts) > 1 else ''
                documents[doc_id] = {'id': doc_id, 'title': title, 'abstract': '', 'entities': []}
                
            elif '|a|' in line:
                # Abstract line
                parts = line.split('|a|')
                doc_id = parts[0]
                abstract = parts[1] if len(parts) > 1 else ''
                if doc_id in documents:
                    documents[doc_id]['abstract'] = abstract
                    
            else:
                # Entity line
                parts = line.split('\t')
                if len(parts) >= 5:
                    doc_id = parts[0]
                    start = int(parts[1])
                    end = int(parts[2])
                    entity_text = parts[3]
                    entity_type = parts[4]  # Disease or Chemical
                    
                    if doc_id in documents:
                        documents[doc_id]['entities'].append((start, end, entity_type))
    
    # Combine title and abstract, and adjust entity positions for abstract
    results = []
    for doc_id, doc in documents.items():
        title = doc['title']
        abstract = doc['abstract']
        # Combined text: title + space + abstract
        combined_text = title + ' ' + abstract if abstract else title
        
        # Adjust entity positions (entities in abstract need +len(title)+1 offset)
        adjusted_entities = []
        title_len = len(title)
        
        for start, end, etype in doc['entities']:
            # Determine if entity is in title or abstract
            if start < title_len:
                # Entity in title, no adjustment needed
                adjusted_entities.append((start, end, etype))
            else:
                # Entity in abstract - account for the space we added
                # Actually, in BC5CDR, positions are relative to title+abstract combined
                adjusted_entities.append((start, end, etype))
        
        results.append({
            'id': doc_id,
            'text': combined_text,
            'entities': adjusted_entities
        })
    
    return results

In [5]:
def text_to_bio_labels(text, entities, tokenizer):
    """
    Convert text with entity annotations to BIO labels aligned with tokenizer output.
    
    Args:
        text: The input text
        entities: List of (start, end, entity_type) tuples
        tokenizer: HuggingFace tokenizer
        
    Returns:
        tokens: List of tokens
        labels: List of BIO labels (as integers)
    """
    # Tokenize the text
    encoding = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=512)
    tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
    offsets = encoding['offset_mapping']
    
    # Initialize all labels as 'O'
    labels = [LABEL2ID['O']] * len(tokens)
    
    # Create character-level entity map
    entity_map = {}  # char_idx -> entity_type
    for start, end, etype in entities:
        for i in range(start, end):
            if i < len(text):
                entity_map[i] = etype
    
    # Assign BIO labels to tokens
    for i, (tok_start, tok_end) in enumerate(offsets):
        # Skip special tokens ([CLS], [SEP], etc.)
        if tok_start == tok_end:
            labels[i] = LABEL2ID['O']
            continue
        
        # Check if any character in this token is part of an entity
        entity_type = None
        for char_idx in range(tok_start, min(tok_end, len(text))):
            if char_idx in entity_map:
                entity_type = entity_map[char_idx]
                break
        
        if entity_type:
            # Determine if this is B- or I-
            if tok_start == 0 or (tok_start - 1) not in entity_map or entity_map.get(tok_start - 1) != entity_type:
                # Beginning of entity
                labels[i] = LABEL2ID[f'B-{entity_type}']
            else:
                # Inside entity
                labels[i] = LABEL2ID[f'I-{entity_type}']
    
    return tokens, labels, encoding

In [6]:
# Demo: Show BIO encoding example
demo_text = "Tricuspid valve regurgitation and lithium carbonate toxicity in a newborn infant."
demo_entities = [
    (0, 29, 'Disease'),    # Tricuspid valve regurgitation
    (34, 51, 'Chemical'),  # lithium carbonate
    (52, 60, 'Disease')    # toxicity
]

# Load tokenizer for demo
demo_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

tokens, labels, _ = text_to_bio_labels(demo_text, demo_entities, demo_tokenizer)

print("Demo BIO Encoding:")
print("-" * 60)
for tok, lab in zip(tokens, labels):
    if tok not in ['[CLS]', '[SEP]']:
        print(f"{tok:20} -> {ID2LABEL[lab]}")

Demo BIO Encoding:
------------------------------------------------------------
tri                  -> B-Disease
##cus                -> I-Disease
##pid                -> I-Disease
valve                -> I-Disease
reg                  -> I-Disease
##urg                -> I-Disease
##itation            -> I-Disease
and                  -> O
lithium              -> B-Chemical
carbonate            -> I-Chemical
toxicity             -> B-Disease
in                   -> O
a                    -> O
newborn              -> O
infant               -> O
.                    -> O


## 4. Create Dataset Class

In [7]:
class NERDataset(Dataset):
    """Dataset for NER with BIO labels."""
    
    def __init__(self, documents, tokenizer, max_length=512):
        """
        Args:
            documents: List of {'text': str, 'entities': [(start, end, type), ...]}
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length
        """
        self.documents = documents
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.processed_data = self._process_documents()
        
    def _process_documents(self):
        """Process all documents and create tokenized inputs with labels."""
        processed = []
        
        for doc in self.documents:
            text = doc['text']
            entities = doc['entities']
            
            # Tokenize and get BIO labels
            tokens, labels, encoding = text_to_bio_labels(
                text, entities, self.tokenizer
            )
            
            processed.append({
                'input_ids': encoding['input_ids'],
                'attention_mask': encoding['attention_mask'],
                'labels': labels
            })
            
        return processed
    
    def __len__(self):
        return len(self.processed_data)
    
    def __getitem__(self, idx):
        item = self.processed_data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(item['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(item['labels'], dtype=torch.long)
        }

## 5. Load and Prepare Data

If you have the BC5CDR dataset, place the files in the data directory. Otherwise, we'll create a sample dataset for demonstration.

In [8]:
# Option 1: Load BC5CDR dataset if available
# Uncomment and modify paths if you have the dataset

# train_docs = parse_bc5cdr_file('./data/CDR_DevelopmentSet.PubTator.txt')
# test_docs = parse_bc5cdr_file('./data/CDR_TestSet.PubTator.txt')

# Option 2: Create sample data for demonstration
sample_documents = [
    {
        'id': '1',
        'text': 'Tricuspid valve regurgitation and lithium carbonate toxicity in a newborn infant.',
        'entities': [(0, 29, 'Disease'), (34, 51, 'Chemical'), (52, 60, 'Disease')]
    },
    {
        'id': '2', 
        'text': 'Aspirin is commonly used to treat headaches and reduce fever.',
        'entities': [(0, 7, 'Chemical'), (34, 43, 'Disease'), (54, 59, 'Disease')]
    },
    {
        'id': '3',
        'text': 'Diabetes mellitus is a chronic disease that affects blood sugar levels.',
        'entities': [(0, 17, 'Disease'), (52, 63, 'Chemical')]
    },
    {
        'id': '4',
        'text': 'Ibuprofen and acetaminophen are effective painkillers for arthritis.',
        'entities': [(0, 9, 'Chemical'), (14, 27, 'Chemical'), (57, 66, 'Disease')]
    },
    {
        'id': '5',
        'text': 'Hypertension treatment often involves beta blockers and diuretics.',
        'entities': [(0, 12, 'Disease'), (37, 51, 'Chemical'), (56, 67, 'Chemical')]
    },
    {
        'id': '6',
        'text': 'The patient was diagnosed with pneumonia and treated with antibiotics.',
        'entities': [(32, 41, 'Disease'), (58, 69, 'Chemical')]
    },
    {
        'id': '7',
        'text': 'Methotrexate is used for cancer treatment and rheumatoid arthritis.',
        'entities': [(0, 12, 'Chemical'), (26, 32, 'Disease'), (37, 57, 'Disease')]
    },
    {
        'id': '8',
        'text': 'Insulin injections help manage diabetes type 1 and type 2.',
        'entities': [(0, 7, 'Chemical'), (32, 47, 'Disease'), (52, 60, 'Disease')]
    }
]

# Split into train and test
train_docs, test_docs = train_test_split(sample_documents, test_size=0.25, random_state=42)

print(f"Training documents: {len(train_docs)}")
print(f"Test documents: {len(test_docs)}")

Training documents: 6
Test documents: 2


## 6. Load PubMedBERT Model

In [9]:
# Load PubMedBERT tokenizer and model
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
num_labels = len(LABEL_LIST)

print(f"Loading model: {model_name}")
print(f"Number of labels: {num_labels}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BertForTokenClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    use_safetensors=True  # 使用 safetensors 格式避免 torch.load 安全问题
)

model = model.to(device)
print("Model loaded successfully!")

Loading model: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Number of labels: 5


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	tho

Model loaded successfully!


In [10]:
# Create datasets
train_dataset = NERDataset(train_docs, tokenizer)
test_dataset = NERDataset(test_docs, tokenizer)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 6
Test samples: 2


## 7. Define Metrics and Training Functions

In [11]:
def compute_metrics(pred):
    """
    Compute NER metrics using seqeval.
    """
    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=2)
    
    # Remove special tokens and convert to label strings
    true_labels = []
    true_preds = []
    
    for label_seq, pred_seq in zip(labels, preds):
        temp_labels = []
        temp_preds = []
        
        for l, p in zip(label_seq, pred_seq):
            if l != -100:  # Ignore special tokens
                temp_labels.append(ID2LABEL[l])
                temp_preds.append(ID2LABEL[p])
                
        if temp_labels:
            true_labels.append(temp_labels)
            true_preds.append(temp_preds)
    
    # Compute metrics
    results = {
        'precision': precision_score(true_labels, true_preds),
        'recall': recall_score(true_labels, true_preds),
        'f1': f1_score(true_labels, true_preds)
    }
    
    return results

In [12]:
# Data collator for padding
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## 8. Train the Model

In [13]:
# Training arguments
training_args = TrainingArguments(
    output_dir='./ner_results',
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    disable_tqdm=False,  # 禁用 notebook progress callback 避免状态问题
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [14]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,1.750022,0.000000,0.000000,0.000000
2,No log,1.742411,0.000000,0.000000,0.000000
3,No log,1.727419,0.000000,0.000000,0.000000
4,No log,1.705098,0.000000,0.000000,0.000000
5,No log,1.675545,0.000000,0.000000,0.000000
6,No log,1.639241,0.000000,0.000000,0.000000
7,No log,1.596646,0.000000,0.000000,0.000000
8,No log,1.547932,0.100000,0.200000,0.133333
9,No log,1.494011,0.111111,0.200000,0.142857
10,1.641228,1.435728,0.000000,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=10, training_loss=1.6412275314331055, metrics={'train_runtime': 96.9859, 'train_samples_per_second': 0.619, 'train_steps_per_second': 0.103, 'total_flos': 428701618800.0, 'train_loss': 1.6412275314331055, 'epoch': 10.0})

## 9. Evaluate the Model

In [15]:
# 使用 predict 直接评估（避免 notebook callback 状态问题）
print("Evaluating model...")
predictions = trainer.predict(test_dataset)

# 计算评估指标
eval_loss = predictions.metrics.get('test_loss', 0)
print(f"\nEvaluation Loss: {eval_loss:.4f}")

# 使用 compute_metrics 计算指标
from types import SimpleNamespace
pred_obj = SimpleNamespace(
    predictions=predictions.predictions,
    label_ids=predictions.label_ids
)
metrics = compute_metrics(pred_obj)
print("\nEvaluation Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

Evaluating model...



Evaluation Loss: 1.4935

Evaluation Metrics:
  precision: 0.1111
  recall: 0.2000
  f1: 0.1429


In [16]:
# Detailed classification report
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=2)
labels = predictions.label_ids

# Convert to seqeval format
true_labels = []
true_preds = []

for label_seq, pred_seq in zip(labels, preds):
    temp_labels = []
    temp_preds = []
    
    for l, p in zip(label_seq, pred_seq):
        if l != -100:
            temp_labels.append(ID2LABEL[l])
            temp_preds.append(ID2LABEL[p])
            
    if temp_labels:
        true_labels.append(temp_labels)
        true_preds.append(temp_preds)

print("\n" + "="*60)
print("Classification Report:")
print("="*60)
print(classification_report(true_labels, true_preds))


Classification Report:
              precision    recall  f1-score   support

    Chemical       0.17      0.50      0.25         2
     Disease       0.00      0.00      0.00         3

   micro avg       0.11      0.20      0.14         5
   macro avg       0.08      0.25      0.12         5
weighted avg       0.07      0.20      0.10         5



## 10. Test on New Text

In [17]:
def predict_ner(text, model, tokenizer, id2label):
    """
    Predict named entities in text.
    
    Returns:
        List of (token, label) tuples
    """
    model.eval()
    
    # Tokenize
    encoding = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    # Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)
    
    # Convert to labels
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    labels = [id2label[p.item()] for p in predictions[0]]
    
    # Filter out special tokens
    results = []
    for tok, lab in zip(tokens, labels):
        if tok not in ['[CLS]', '[SEP]', '[PAD]']:
            results.append((tok, lab))
    
    return results

In [18]:
# Test on sample medical texts
test_texts = [
    "The patient was prescribed metformin for diabetes management.",
    "Acetaminophen overdose can cause liver damage and failure.",
    "Hypertension and hyperlipidemia are common cardiovascular risk factors."
]

print("\n" + "="*60)
print("Predictions on New Text:")
print("="*60)

for text in test_texts:
    print(f"\nText: {text}")
    print("-" * 40)
    results = predict_ner(text, model, tokenizer, ID2LABEL)
    
    current_entity = []
    current_type = None
    
    for token, label in results:
        if label.startswith('B-'):
            # New entity starts
            if current_entity:
                print(f"  {' '.join(current_entity)}: {current_type}")
            current_entity = [token]
            current_type = label[2:]
        elif label.startswith('I-') and current_entity:
            current_entity.append(token)
        else:
            if current_entity:
                print(f"  {' '.join(current_entity)}: {current_type}")
                current_entity = []
                current_type = None
    
    if current_entity:
        print(f"  {' '.join(current_entity)}: {current_type}")


Predictions on New Text:

Text: The patient was prescribed metformin for diabetes management.
----------------------------------------

Text: Acetaminophen overdose can cause liver damage and failure.
----------------------------------------
  can: Disease
  .: Disease

Text: Hypertension and hyperlipidemia are common cardiovascular risk factors.
----------------------------------------


## 11. Save the Model

In [19]:
# Save model and tokenizer
save_path = './medical_ner_model'

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./medical_ner_model


In [20]:
# Load saved model
loaded_model = BertForTokenClassification.from_pretrained(save_path, use_safetensors=True)
loaded_tokenizer = AutoTokenizer.from_pretrained(save_path)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully!


## 12. Summary

In this lab, we:

1. **Learned BIO encoding** - Converting entity spans to token-level labels (B-/I-/O)
2. **Parsed BC5CDR format** - Medical NER dataset with diseases and chemicals
3. **Used PubMedBERT** - Domain-specific pre-trained model for medical texts
4. **Fine-tuned the model** - Using Hugging Face Transformers Trainer
5. **Evaluated with seqeval** - Standard NER evaluation metrics (precision, recall, F1)

### Next Steps

- Download the full BC5CDR dataset from [BioCreative](https://biocreative.bioinformatics.udel.edu/tasks/biocreative-v/track-3-cdr/)
- Apply the model to analyze medical literature (e.g., COVID-19 papers)
- Experiment with different transformer models (BioBERT, ClinicalBERT)
- Try different learning rates and batch sizes for better performance